# Tahap 3 — Training & Evaluasi: VGG16

**Ruang lingkup notebook ini: VGG16 saja.** Tugas ini berkelompok — anggota lain melatih arsitektur
berbeda (mis. EfficientNet-B0), dan hasilnya digabung di tahap perbandingan.

Seluruh logika training ada di modul **`training.py`** yang arsitektur-agnostik, bukan di notebook ini.
Perbandingan antar arsitektur hanya sahih kalau split, preprocessing, hyperparameter, dan cara menghitung
metriknya **identik**. Karena itu notebook ini sama persis dengan `train_efficientnet.ipynb`; satu-satunya
perbedaan di kode adalah nama model:

```python
model = tr.build_model("vgg16", num_classes=31)   # ganti nama arsitektur saja
```

Hasil ditulis ke `results/vgg16/` dalam format seragam, siap digabung dengan hasil arsitektur lain.

**Prasyarat:** `prepare_data.ipynb` (split + dedup) dan `preprocessing.ipynb` (Tahap 2) sudah dijalankan,
dan training sudah dijalankan dari terminal (`python train_vgg16.py`) sehingga `results/vgg16/` dan
`models/vgg16_best.pt` tersedia.

## 1. Setup

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

import data_pipeline as dp
import training as tr

MODEL_NAME = "vgg16"

# Palet: biru = train, oranye = validation. Dipakai konsisten di semua grafik.
C_TRAIN, C_VAL, C_ACCENT = "#2a78d6", "#eb6834", "#1baf7a"
C_GRID, C_TEXT = "#d8d7d2", "#52514e"
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True, "grid.color": C_GRID, "grid.linewidth": 0.6,
    "axes.edgecolor": C_GRID, "axes.labelcolor": C_TEXT,
    "xtick.color": C_TEXT, "ytick.color": C_TEXT,
    "axes.spines.top": False, "axes.spines.right": False,
})

dp.seed_everything(dp.SEED)
DEVICE = dp.get_device()
print(f"Device: {DEVICE}")
print(f"Torch : {torch.__version__}")

## 2. Tentang VGG16

Bagian ini untuk bagian "Penjelasan arsitektur" di laporan.

**Ide utama: kedalaman dengan kernel kecil.** Simonyan & Zisserman (2014, Visual Geometry Group, Oxford)
menunjukkan bahwa akurasi naik ketika jaringan diperdalam, asalkan seluruh konvolusi memakai kernel kecil
3×3. Dua konvolusi 3×3 yang ditumpuk punya receptive field setara satu konvolusi 5×5, dan tiga tumpuk
setara 7×7 — tetapi dengan parameter lebih sedikit (tiga lapis 3×3 butuh `27C²` bobot, satu lapis 7×7
butuh `49C²`) dan dengan tiga non-linearitas ReLU, bukan satu. Itu sebabnya VGG memilih tumpukan 3×3 yang
seragam, bukan kernel besar.

**Susunan.** VGG16 punya 16 layer berbobot: 13 konvolusi 3×3 (stride 1, padding 1) yang dikelompokkan
menjadi 5 blok dengan 64 → 128 → 256 → 512 → 512 filter, dipisah max-pooling 2×2 yang tiap kali memangkas
resolusi jadi separuh, lalu 3 layer fully connected (4096 → 4096 → jumlah kelas) dengan dropout 0,5. Input
224×224 berakhir sebagai peta fitur 7×7×512 = 25.088 nilai yang diratakan ke layer FC. Di notebook ini
hanya layer FC terakhir yang diganti (1000 → 31 kelas); seluruh jaringan di-fine-tune dari bobot ImageNet.

**Di mana parameternya.** Seluruh konvolusi hanya ±14,7 M parameter. Layer FC pertama saja
(25.088 × 4096) sudah ±102,8 M — sekitar tiga perempat dari total 134,4 M. Jadi ukuran VGG16 yang besar
bukan karena kedalamannya, melainkan karena kepala klasifikasinya yang lebar.

**Biaya komputasi.** ±15,5 GFLOPs per citra 224×224 (angka torchvision): kira-kira 40 kali EfficientNet-B0
(±0,39) dan 4 kali ResNet-50 (±4,1). Semua konvolusinya biasa (bukan depthwise separable) dan resolusi
tinggi dipertahankan di blok awal, sehingga mahal untuk training maupun inferensi.

**Yang tidak dimiliki.** VGG16 klasik tidak punya batch normalization maupun skip connection (varian
`vgg16_bn` punya BN). Dilatih dari nol, jaringan sedalam ini sulit dioptimasi; dengan bobot pretrained dan
lr kecil (1e-4) masalah itu tidak muncul di sini.

**Relevansi untuk dataset ini.** 31 kelas dengan ±136 citra train per kelas tergolong dataset kecil, dan
134 M parameter memberi kapasitas jauh melebihi kebutuhan — risiko overfitting lebih tinggi dibanding
model kecil seperti EfficientNet-B0 (4 M). Hipotesis ini diuji lewat gap train-val di Bagian 4.1 dan
perbandingan lintas arsitektur di tahap kelompok.

In [ ]:
# Konteks ukuran arsitektur (bobot tidak diunduh, hanya struktur)
baris = []
for name in tr.ARCHITECTURES:
    m = tr.build_model(name, num_classes=31, pretrained=False)
    info = tr.count_parameters(m)
    baris.append({"arsitektur": name,
                  "params (M)": round(info["total_params"] / 1e6, 1),
                  "ukuran (MB)": round(info["size_mb"], 1)})
    del m

konteks = pd.DataFrame(baris).sort_values("params (M)")
print(konteks.to_string(index=False))
print("\n(Baris selain vgg16 hanya konteks ukuran; pelatihannya bagian anggota kelompok lain.)")

## 3. Data & Model

In [ ]:
CONFIG = tr.TrainConfig(
    epochs=5,
    lr=1e-4,
    weight_decay=1e-4,
    batch_size=32,
    patience=5,
    augment="basic",
    use_class_weights=True,
)

loaders, ds = dp.build_dataloaders(
    # num_workers=2 dan batch_size=32 disamakan dengan train_efficientnet.py
    # supaya protokol latih identik antar arsitektur.
    batch_size=CONFIG.batch_size, num_workers=2, augment=CONFIG.augment
)
classes = ds["train"].classes

print(f"train/val/test : {len(ds['train'])}/{len(ds['val'])}/{len(ds['test'])}")
print(f"kelas          : {len(classes)}")
print(f"batch per epoch: {len(loaders['train'])}")
print(f"\nHyperparameter (harus sama untuk semua arsitektur yang dibandingkan):")
for k, v in vars(CONFIG).items():
    print(f"  {k:20s}: {v}")

In [ ]:
model = tr.build_model(MODEL_NAME, num_classes=len(classes), pretrained=True)
info = tr.count_parameters(model)

print(f"{MODEL_NAME}: {info['total_params']:,} parameter ({info['total_params']/1e6:.2f} M), "
      f"{info['size_mb']:.1f} MB")
print(f"\nLayer klasifikasi setelah diganti:\n{model.classifier}")

# Seluruh jaringan di-fine-tune, bukan hanya head. Dengan ~4200 citra train,
# fine-tuning penuh memberi akurasi lebih tinggi daripada membekukan backbone,
# dan lr kecil (1e-4) menjaga fitur pretrained tidak rusak di epoch awal.
print(f"\nParameter yang dilatih: {info['trainable_params']:,} (seluruh jaringan, fine-tuning penuh)")

## 4. Training

Konfigurasi: AdamW + cosine annealing, `CrossEntropyLoss` berbobot kelas (menangani imbalance 1.73x dari
Tahap 1), early stopping pada **val macro-F1** dengan patience 5.

Dipilih macro-F1, bukan akurasi, sebagai kriteria model terbaik: dengan kelas yang tidak seimbang,
akurasi bisa naik hanya karena model membaik di kelas besar sementara kelas kecil seperti `asam jawa`
justru memburuk. Macro-F1 memberi bobot sama ke tiap kelas.

**Training dijalankan dari terminal, bukan dari sel notebook:**

```bash
python train_vgg16.py                 # progres tampil live
tail -f logs/vgg16.log                # atau pantau dari terminal lain
```

Alasannya praktis — output sel notebook baru terlihat setelah selnya selesai, sehingga training belasan
menit berjalan tanpa tanda kehidupan. Skrip mencetak progres per 20 batch lengkap dengan estimasi sisa
waktu. Notebook ini lalu memuat checkpoint dan history hasil run tersebut, jadi grafik di bawah dibuat
tanpa melatih ulang.

Set `RETRAIN = True` kalau memang ingin melatih dari sel notebook (±17 menit untuk VGG16 di M1 Pro; ini menimpa hasil dan checkpoint yang ada) — progres tiap epoch tampil sebagai
progress bar (batch ke-berapa, loss berjalan, estimasi sisa waktu), lalu satu baris ringkasan per epoch.

In [ ]:
RETRAIN = False   # True = latih ulang dari notebook (progress bar per epoch tampil di output sel)

sudah_ada = (Path("results") / MODEL_NAME / "history.json").exists()

if RETRAIN or not sudah_ada:
    if not sudah_ada:
        print("Hasil training belum ada. Disarankan jalankan dulu dari terminal:\n"
              "    python train_vgg16.py\n"
              "Melanjutkan dengan training dari notebook...\n")
    t0 = time.perf_counter()
    hasil = tr.fit(model, loaders, ds, model_name=MODEL_NAME, config=CONFIG, device=DEVICE)
    print(f"\nTotal waktu training: {(time.perf_counter() - t0)/60:.1f} menit")
else:
    hasil = tr.load_run(MODEL_NAME, model, device=DEVICE)
    print(f"Memuat hasil training tersimpan (dari train_vgg16.py).")
    print(f"Jalankan ulang dengan RETRAIN = True kalau ingin melatih dari awal.\n")

print(f"Epoch dijalankan : {len(hasil['history'])}")
print(f"Epoch terbaik    : {hasil['best_epoch']} (val macro-F1 {hasil['best_val_macro_f1']:.4f})")
print(f"Waktu training   : {hasil['total_train_time_s']/60:.1f} menit")
print(f"Checkpoint       : {hasil['checkpoint']}")

### 4.1 Kurva Belajar

In [ ]:
hist = pd.DataFrame(hasil["history"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].plot(hist["epoch"], hist["train_loss"], color=C_TRAIN, lw=2, label="train")
axes[0].plot(hist["epoch"], hist["val_loss"], color=C_VAL, lw=2, label="validation")
axes[0].set(xlabel="epoch", ylabel="loss", title="Loss")
axes[0].legend(frameon=False)

axes[1].plot(hist["epoch"], hist["train_acc"], color=C_TRAIN, lw=2, label="train")
axes[1].plot(hist["epoch"], hist["val_acc"], color=C_VAL, lw=2, label="validation")
axes[1].set(xlabel="epoch", ylabel="akurasi", title="Akurasi")
axes[1].legend(frameon=False)

axes[2].plot(hist["epoch"], hist["val_macro_f1"], color=C_VAL, lw=2)
best = hasil["best_epoch"]
best_f1 = hasil["best_val_macro_f1"]
axes[2].scatter([best], [best_f1], color=C_VAL, s=60, zorder=5)
axes[2].annotate(f"terbaik: epoch {best}\nF1 {best_f1:.4f}",
                 xy=(best, best_f1), xytext=(6, -28), textcoords="offset points",
                 fontsize=9, color=C_TEXT)
axes[2].set(xlabel="epoch", ylabel="macro-F1", title="Validation macro-F1 (kriteria early stopping)")

plt.tight_layout()
plt.show()

print(hist[["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "val_macro_f1", "train_time_s"]]
      .round(4).to_string(index=False))

gap = hist.loc[hist["epoch"] == best, "train_acc"].iloc[0] - hist.loc[hist["epoch"] == best, "val_acc"].iloc[0]
print(f"\nGap train-val akurasi di epoch terbaik: {gap:+.4f}")
print("Gap besar (>0.10) menandakan overfitting -> pertimbangkan augment='strong'.")

## 5. Evaluasi pada Test Set

Test set baru disentuh **sekali, di sini**, memakai checkpoint terbaik yang dipilih berdasarkan
validation. Memilih model berdasarkan skor test akan membuat angka test tidak lagi menjadi estimasi
jujur terhadap data baru.

In [ ]:
evalres = tr.evaluate(model, loaders, ds, model_name=MODEL_NAME, hasil_fit=hasil, device=DEVICE)
m = evalres["metrics"]

print(f"=== {MODEL_NAME} — hasil test ===\n")
ringkas = pd.DataFrame([
    ("Accuracy", m["accuracy"]),
    ("Top-5 accuracy", m["top5_accuracy"]),
    ("Precision (macro)", m["precision_macro"]),
    ("Recall (macro)", m["recall_macro"]),
    ("F1 (macro)", m["f1_macro"]),
    ("Precision (weighted)", m["precision_weighted"]),
    ("Recall (weighted)", m["recall_weighted"]),
    ("F1 (weighted)", m["f1_weighted"]),
], columns=["metrik", "nilai"])
print(ringkas.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print(f"\nSelisih val macro-F1 ({hasil['best_val_macro_f1']:.4f}) vs test macro-F1 ({m['f1_macro']:.4f}): "
      f"{m['f1_macro'] - hasil['best_val_macro_f1']:+.4f}")
print(f"Hasil disimpan ke: {evalres['output_dir']}")

### 5.1 Metrik per Kelas

In [ ]:
per_class = pd.DataFrame(evalres["per_class"]).sort_values("f1")

fig, ax = plt.subplots(figsize=(9, 9))
y = np.arange(len(per_class))
ax.barh(y, per_class["f1"], color=C_TRAIN, height=0.7)
ax.set_yticks(y)
ax.set_yticklabels(per_class["class"], fontsize=8)
ax.axvline(m["f1_macro"], color=C_VAL, lw=2, ls="--")
ax.annotate(f"macro-F1 {m['f1_macro']:.3f}", xy=(m["f1_macro"], len(per_class) - 1.5),
            xytext=(6, 0), textcoords="offset points", fontsize=9, color=C_VAL)
ax.set(xlabel="F1-score", xlim=(0, 1.02), title=f"F1 per kelas — {MODEL_NAME} (test set)")
ax.grid(axis="y", visible=False)
# Label langsung pada 5 kelas terlemah: bagian yang paling perlu dibaca angkanya
for i in range(5):
    ax.text(per_class["f1"].iloc[i] + 0.012, i, f"{per_class['f1'].iloc[i]:.2f}",
            va="center", fontsize=8, color=C_TEXT)
plt.tight_layout()
plt.show()

print("5 kelas terlemah:")
print(per_class.head(5)[["class", "precision", "recall", "f1", "support"]]
      .round(4).to_string(index=False))
print("\n5 kelas terkuat:")
print(per_class.tail(5)[["class", "precision", "recall", "f1", "support"]]
      .round(4).to_string(index=False))

In [ ]:
# Apakah kelas kecil memang lebih sulit? Relevan karena Tahap 1 menemukan
# imbalance 1.73x setelah dedup.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(per_class["support"], per_class["f1"], color=C_TRAIN, s=45, zorder=3)
for _, r in per_class.head(4).iterrows():
    ax.annotate(r["class"], xy=(r["support"], r["f1"]), xytext=(5, -3),
                textcoords="offset points", fontsize=8, color=C_TEXT)
korelasi = per_class["support"].corr(per_class["f1"])
ax.set(xlabel="jumlah citra test (support)", ylabel="F1-score",
       title=f"F1 vs ukuran kelas (korelasi Pearson r = {korelasi:.2f})")
plt.tight_layout()
plt.show()
print(f"Korelasi support vs F1: r = {korelasi:.3f}")
print("r mendekati 0 berarti kesulitan kelas lebih ditentukan kemiripan visual daripada jumlah data.")

### 5.2 Confusion Matrix

In [ ]:
cm = evalres["confusion_matrix"]
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9.5))
# Sequential satu hue (terang -> gelap) untuk besaran; bukan rainbow.
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=90, fontsize=7)
ax.set_yticklabels(classes, fontsize=7)
ax.set(xlabel="prediksi", ylabel="label sebenarnya",
       title=f"Confusion matrix ternormalisasi per baris — {MODEL_NAME} (test)")
ax.grid(visible=False)
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("proporsi dari kelas sebenarnya", color=C_TEXT)
plt.tight_layout()
plt.show()

# Daftar kebingungan terbesar lebih terbaca daripada mencari sel gelap satu per satu
salah = [(classes[i], classes[j], cm[i, j], cm_norm[i, j])
         for i in range(len(classes)) for j in range(len(classes))
         if i != j and cm[i, j] > 0]
salah.sort(key=lambda t: -t[2])

konfusi = pd.DataFrame(salah[:15], columns=["sebenarnya", "diprediksi", "n", "proporsi"])
print("15 kebingungan terbesar:")
print(konfusi.round(3).to_string(index=False))
print(f"\nTotal salah klasifikasi: {int(cm.sum() - np.trace(cm))} dari {int(cm.sum())} citra test")

### 5.3 Analisis Kesalahan

In [ ]:
probs, preds, trues = evalres["probs"], evalres["preds"], evalres["trues"]
paths = [p for p, _ in ds["test"].samples]

salah_idx = np.where(preds != trues)[0]
conf = probs[np.arange(len(preds)), preds]

print(f"Jumlah salah: {len(salah_idx)} ({len(salah_idx)/len(preds):.2%})")
print(f"Rata-rata confidence saat BENAR : {conf[preds == trues].mean():.4f}")
print(f"Rata-rata confidence saat SALAH : {conf[salah_idx].mean():.4f}")
print("\nModel yang terkalibrasi baik seharusnya kurang yakin ketika salah.")

# Kesalahan paling percaya diri = kandidat label salah atau kelas yang benar-benar mirip
urut = salah_idx[np.argsort(-conf[salah_idx])][:10]
fig, axes = plt.subplots(2, 5, figsize=(15, 6.6))
for ax, i in zip(axes.ravel(), urut):
    ax.imshow(Image.open(paths[i]).convert("RGB"))
    ax.set_title(f"asli: {classes[trues[i]]}\nprediksi: {classes[preds[i]]} ({conf[i]:.2f})",
                 fontsize=8, color=C_TEXT)
    ax.axis("off")
fig.suptitle("10 kesalahan dengan confidence tertinggi")
plt.tight_layout()
plt.show()

In [ ]:
# Prediksi benar dari model terbaik, sesuai permintaan tugas langkah 5
rng = np.random.default_rng(dp.SEED)
benar_idx = np.where(preds == trues)[0]
pilih = rng.choice(benar_idx, 10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6.6))
for ax, i in zip(axes.ravel(), pilih):
    ax.imshow(Image.open(paths[i]).convert("RGB"))
    ax.set_title(f"{classes[preds[i]]}\nconfidence {conf[i]:.2f}", fontsize=8, color=C_TEXT)
    ax.axis("off")
fig.suptitle("Contoh prediksi benar")
plt.tight_layout()
plt.show()

## 6. Efisiensi Komputasi

In [ ]:
eff = pd.DataFrame([
    ("Jumlah parameter", f"{m['total_params']:,} ({m['total_params']/1e6:.2f} M)"),
    ("Ukuran model", f"{m['size_mb']:.1f} MB"),
    ("Epoch dijalankan", f"{m['epochs_run']} (terbaik: epoch {m['best_epoch']})"),
    ("Waktu training total", f"{m['total_train_time_s']/60:.1f} menit"),
    ("Rata-rata waktu/epoch", f"{m['mean_epoch_time_s']:.1f} detik"),
    ("Latensi inferensi", f"{m['infer_latency_ms_per_batch']:.1f} ms/batch "
                          f"({m['infer_latency_ms_per_image']:.2f} ms/citra)"),
    ("Throughput inferensi", f"{m['infer_throughput_img_per_s']:.0f} citra/detik"),
    ("Device", f"{m['device']} — {m['platform']}"),
], columns=["metrik", "nilai"])
print(eff.to_string(index=False))

print(f"\nAkurasi per juta parameter: {m['accuracy'] / (m['total_params']/1e6):.4f}")
print("Angka ini berguna untuk membandingkan efisiensi arsitektur di tahap perbandingan kelompok.")

## 7. Ekspor Hasil untuk Perbandingan Kelompok

In [ ]:
out_dir = Path(evalres["output_dir"])
print(f"Isi {out_dir}:")
for f in sorted(out_dir.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size/1024:8.1f} KB")

print(f"""
Protokol yang dipakai SAMA dengan train_efficientnet.py:
  - data_split/ hasil dedup ({m['n_train']}/{m['n_val']}/{m['n_test']})
  - dp.build_dataloaders() yang sama (224x224, normalisasi ImageNet, augment="{CONFIG.augment}")
  - TrainConfig yang sama (lr={CONFIG.lr}, epochs={CONFIG.epochs}, batch={CONFIG.batch_size})
  - metrik dihitung oleh tr.evaluate() yang sama

CATATAN untuk laporan:
  - VGG16 dilatih pada batch {CONFIG.batch_size} di mesin ini ({m['platform']}, device {m['device']})
    tanpa masalah memori.
  - Waktu training dan latensi inferensi TIDAK apple-to-apple dengan arsitektur yang dilatih di
    mesin lain. Catat spesifikasi mesin masing-masing di laporan, atau ukur ulang di satu laptop.
  - Perbandingan akurasi/F1 tetap sahih hanya kalau split data identik (cek manifest.csv).
""")

In [ ]:
# Ringkasan satu baris, siap digabung jadi tabel perbandingan di laporan
ringkasan_baris = {
    "arsitektur": MODEL_NAME,
    "params_M": round(m["total_params"] / 1e6, 2),
    "accuracy": round(m["accuracy"], 4),
    "precision_macro": round(m["precision_macro"], 4),
    "recall_macro": round(m["recall_macro"], 4),
    "f1_macro": round(m["f1_macro"], 4),
    "train_time_min": round(m["total_train_time_s"] / 60, 1),
    "infer_ms_per_image": round(m["infer_latency_ms_per_image"], 2),
}
print(pd.DataFrame([ringkasan_baris]).to_string(index=False))

Path("results").mkdir(exist_ok=True)
pd.DataFrame([ringkasan_baris]).to_csv(f"results/{MODEL_NAME}/summary_row.csv", index=False)
print(f"\nDisimpan: results/{MODEL_NAME}/summary_row.csv")

## 8. Ringkasan

Poin-poin ini yang masuk ke laporan. Angka yang sudah ada berasal dari log training; bagian dalam
kurung siku diisi setelah melihat output di atas.

- **Performa.** Test accuracy 0,9359 dan macro-F1 0,9354. Val macro-F1 terbaik 0,9358 (epoch 5), jadi
  selisih val-test hanya −0,0004: tidak ada tanda model ter-overfit ke validation lewat early stopping.
- **Overfitting.** Di epoch 5 train acc 0,9575 vs val acc 0,9358 (gap +0,022) — kecil untuk 134 M
  parameter. Val loss masih turun (0,2352 → 0,2018 dari epoch 4 ke 5) dan epoch terbaik adalah epoch
  terakhir, artinya model belum konvergen penuh; batas 5 epoch (sesuai protokol kelompok) membatasi hasil.
- **Kelas tersulit.** [isi dari 5 kelas ber-F1 terendah (Bagian 5.1) dan tabel kebingungan terbesar
  (Bagian 5.2). Periksa apakah pasangan yang tertukar memang mirip secara visual.]
- **Keterbatasan data (Tahap 1).** Citra beresolusi kecil yang di-upsample ke 224 px, dan 479 citra
  (7,36%) dibuang saat dedup, paling banyak dari kelas `asam jawa` (89 citra). Perlu disebut di laporan.
- **Efisiensi.** 134,4 M parameter dan 537,6 MB (tabel Bagian 2) — model terbesar di kelompok ini.
  Training ±17 menit (±3,3 menit/epoch) di M1 Pro. [isi latensi dan throughput dari Bagian 6.]
  Waktu antar-mesin tidak sebanding; lihat catatan di Bagian 7.
- **Untuk tahap perbandingan.** Yang diuji: apakah kapasitas besar (134 M) memberi keunggulan akurasi
  pada dataset kecil (~136 citra train/kelas) dibanding model kecil. Jawabannya baru sahih setelah
  split data kedua model terbukti identik dan selisih akurasi dibandingkan dengan derau antar-run.